# Data Preprocessing Notebook

Steps:
1. Load raw synthetic dataset
2. Validate required schema
3. Split into train/test
4. One-hot encode and align columns
5. Scale physical columns with MinMaxScaler
6. Save raw/scaled splits and scaler artifact

In [1]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
SPLIT_PREFIX = "dataset_optimization_cereal_co2"
PHYSICAL_COLUMNS = ["generated_volume_tons", "moisture_pct", "process_temperature_c"]

RAW_REQUIRED_COLUMNS = [
    "subproduct_type",
    "season",
    "generated_volume_tons",
    "moisture_pct",
    "process_temperature_c",
    "reuse_strategy",
    "co2_emissions_kg",
    "co2_per_ton",
]

EXPECTED_FEATURE_COLUMNS = [
    "generated_volume_tons",
    "moisture_pct",
    "process_temperature_c",
    "subproduct_type_Husk",
    "subproduct_type_Straw",
    "subproduct_type_Silo dust",
    "subproduct_type_Bran",
    "season_Rainy",
    "season_Dry",
]

In [3]:
def validate_input_schema(df: pd.DataFrame) -> None:
    missing = [col for col in RAW_REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for preprocessing: {missing}")


def one_hot_and_align(train_df: pd.DataFrame, test_df: pd.DataFrame):
    categorical_columns = ["subproduct_type", "season"]
    train_encoded = pd.get_dummies(train_df, columns=categorical_columns)
    test_encoded = pd.get_dummies(test_df, columns=categorical_columns)

    for column in EXPECTED_FEATURE_COLUMNS:
        if column not in train_encoded.columns and column not in PHYSICAL_COLUMNS:
            train_encoded[column] = 0
        if column not in test_encoded.columns and column not in PHYSICAL_COLUMNS:
            test_encoded[column] = 0

    train_columns = EXPECTED_FEATURE_COLUMNS + [
        col for col in train_encoded.columns if col not in EXPECTED_FEATURE_COLUMNS
    ]
    train_encoded = train_encoded.reindex(columns=train_columns, fill_value=0)
    test_encoded = test_encoded.reindex(columns=train_columns, fill_value=0)
    return train_encoded, test_encoded


def scale_physical_columns(train_df: pd.DataFrame, test_df: pd.DataFrame):
    scaler = MinMaxScaler()

    train_scaled = train_df.copy()
    test_scaled = test_df.copy()

    train_scaled[PHYSICAL_COLUMNS] = scaler.fit_transform(train_scaled[PHYSICAL_COLUMNS])
    test_scaled[PHYSICAL_COLUMNS] = scaler.transform(test_scaled[PHYSICAL_COLUMNS])

    train_scaled = train_scaled.reindex(
        columns=EXPECTED_FEATURE_COLUMNS + [col for col in train_scaled.columns if col not in EXPECTED_FEATURE_COLUMNS],
        fill_value=0,
    )
    test_scaled = test_scaled.reindex(
        columns=EXPECTED_FEATURE_COLUMNS + [col for col in test_scaled.columns if col not in EXPECTED_FEATURE_COLUMNS],
        fill_value=0,
    )

    return train_scaled, test_scaled, scaler

In [4]:
# Resolve paths from notebooks/ working dir or project root
input_candidates = [
    Path("../data/processed/dataset_optimization_cereal_co2.csv"),
    Path("data/processed/dataset_optimization_cereal_co2.csv"),
]

output_candidates = [
    Path("../data/split"),
    Path("data/split"),
]

input_path = next((p for p in input_candidates if p.exists()), None)
if input_path is None:
    raise FileNotFoundError("Could not find dataset_optimization_cereal_co2.csv")

output_dir = output_candidates[0]
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input dataset: {input_path.resolve()}")
print(f"Output directory: {output_dir.resolve()}")

Input dataset: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\processed\dataset_optimization_cereal_co2.csv
Output directory: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split


In [5]:
df = pd.read_csv(input_path)
validate_input_schema(df)

train_raw, test_raw = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train_raw = train_raw.reset_index(drop=True)
test_raw = test_raw.reset_index(drop=True)

train_encoded, test_encoded = one_hot_and_align(train_raw, test_raw)
train_scaled, test_scaled, scaler = scale_physical_columns(train_encoded, test_encoded)

print(f"Original shape: {df.shape}")
print(f"Train raw shape: {train_raw.shape}")
print(f"Test raw shape: {test_raw.shape}")
print(f"Train scaled shape: {train_scaled.shape}")
print(f"Test scaled shape: {test_scaled.shape}")

Original shape: (50000, 8)
Train raw shape: (40000, 8)
Test raw shape: (10000, 8)
Train scaled shape: (40000, 12)
Test scaled shape: (10000, 12)


In [6]:
train_raw_path = output_dir / f"{SPLIT_PREFIX}_train_raw.csv"
test_raw_path = output_dir / f"{SPLIT_PREFIX}_test_raw.csv"
train_scaled_path = output_dir / f"{SPLIT_PREFIX}_train_scaled.csv"
test_scaled_path = output_dir / f"{SPLIT_PREFIX}_test_scaled.csv"
scaler_path = output_dir / f"{SPLIT_PREFIX}_scaler.joblib"

train_raw.to_csv(train_raw_path, index=False)
test_raw.to_csv(test_raw_path, index=False)
train_scaled.to_csv(train_scaled_path, index=False)
test_scaled.to_csv(test_scaled_path, index=False)
joblib.dump(scaler, scaler_path)

print("Saved preprocessing artifacts:")
print(f"- {train_raw_path.resolve()}")
print(f"- {test_raw_path.resolve()}")
print(f"- {train_scaled_path.resolve()}")
print(f"- {test_scaled_path.resolve()}")
print(f"- {scaler_path.resolve()}")

Saved preprocessing artifacts:
- C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_train_raw.csv
- C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_test_raw.csv
- C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_train_scaled.csv
- C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_test_scaled.csv
- C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_scaler.joblib
